In [ ]:
import os
import numpy as np
import tensorflow as tf
import kagglehub
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [ ]:
# Download Dataset

path = kagglehub.dataset_download("viictte/wikipedia-txt-dataset")
print("Dataset path:", path)

In [ ]:
#  Read Text Files (LIMIT DATA)
all_text = ""

MAX_FILES = 5          
MAX_CHARS = 2_000_000  

txt_files = [f for f in os.listdir(path) if f.endswith(".txt")]
print("Total txt files found:", len(txt_files))

count_files = 0

for file_name in txt_files:
    file_path = os.path.join(path, file_name)

    with open(file_path, "r", encoding="utf-8") as f:
        data = f.read()

    all_text += " " + data

    count_files += 1
    print(f"Loaded {count_files} file(s): {file_name}")

    if count_files >= MAX_FILES or len(all_text) >= MAX_CHARS:
        break

In [ ]:
# Basic cleaning
text = all_text.lower().replace("\n", " ")
print("Total characters loaded:", len(text))

In [ ]:
# Tokenization

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

In [ ]:
#  Create Sequences

input_sequences = []
sentences = text.split(".")  

for line in sentences:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

max_sequence_len = max([len(seq) for seq in input_sequences])

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Max sequence length:", max_sequence_len)

In [ ]:
#  Build LSTM Model

model = Sequential([
    Embedding(total_words, 128, input_length=max_sequence_len - 1),
    LSTM(256, return_sequences=True),
    Dropout(0.2),
    LSTM(256),
    Dense(256, activation="relu"),
    Dense(total_words, activation="softmax")
])

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

In [ ]:
#  Train Model

model.fit(X, y, epochs=10, batch_size=128)

In [ ]:
#  Save Model + Tokenizer

model.save("text_generation_lstm.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("maxlen.pkl", "wb") as f:
    pickle.dump(max_sequence_len, f)

print("Model + tokenizer saved successfully!")